# snm3C Mapping Summary

This notebook provides a quick overview of key mapping metrics from the hisat3n pipeline. You can customize it as needed.

## Parameters

## Prepare

In [ ]:
# parameters
output_dir = ''
plate_col = 'Plate'
color_quantile = (0.05, 0.95)

### Load

In [ ]:
import pathlib
import pandas as pd
from cemba_data.utilities import get_configuration

output_dir = pathlib.Path(output_dir)
mapping_summary = pd.read_csv(output_dir / 'stats/MappingSummary.csv.gz', index_col=0)
config = get_configuration(output_dir / 'mapping_config.ini')

In [ ]:
mapping_summary.columns

### Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from cemba_data.summary import cutoff_vs_cell_remain, plot_on_plate


def distplot_and_plate_view(data, hue, color_quantile=color_quantile, config=config):
    fig1, (vmin, vmax) = cutoff_vs_cell_remain(data=data[hue].dropna(), 
                                               bins=50, kde=False,
                                               xlim_quantile=color_quantile)

    fig2, plate_names, plate_datas = plot_on_plate(
        data=data,
        hue=hue,
        groupby=plate_col,
        vmin=vmin,
        vmax=vmax,
        aggregation_func=lambda i: i.mean())
    
    fig3, ax = plt.subplots(figsize=(data[plate_col].unique().size * 2, 4))
    plate_hue_name = 'Plate'
    sns.boxenplot(data=data, x=plate_col, y=hue, palette='hls', 
                  ax=ax, hue=plate_hue_name)
#    ax.set_ylim(vmin, vmax)
    ax.xaxis.set_tick_params(rotation=90)
    sns.despine(ax=ax)
    return

In [ ]:
# plot defaults
sns.set_context(context='notebook')
plt.rc('figure', dpi=150)

## Summary

In [ ]:
mccc_cutoff = 0.03
high_mccc = mapping_summary['mCCCFrac'] > mccc_cutoff

miseq_guess = mapping_summary['TrimmedReadPairs'].mean() < 50000
reads_cutoff = 200 if miseq_guess else 500000
low_reads = mapping_summary['UniqueAlignFinalReads'] < reads_cutoff

n_cell = mapping_summary.shape[0]
n_plate = mapping_summary['Plate'].unique().size
total_wells = n_plate * 384

In [ ]:
print(f"""
This library seems to be a {'MiSeq' if miseq_guess else 'NovaSeq'} library.

Cells
    {n_plate}\t plates
    {total_wells}\t wells (total cell number in theory)
    {n_cell} ({n_cell / total_wells * 100:.1f}%)\t cells having mapping metric
    {high_mccc.sum()} ({high_mccc.sum() / total_wells * 100:.1f}%)\tcells having high mCCC frac (> {mccc_cutoff})
    {low_reads.sum()} ({low_reads.sum() / total_wells * 100:.1f}%)\tcells having low UniqueAlignFinalReads (< {reads_cutoff})

Reads
    {mapping_summary['TrimmedReadPairs'].sum()*2:.0f}\tTotal Trimmed Reads input to mapping (R1 & R2)
    {mapping_summary['TrimmedReadPairs'].mean()*2:.0f}\tAverage Trimmed Reads per cell (R1 & R2)
    {mapping_summary['UniqueAlignFinalReads'].sum():.0f}\tTotal Final Unique Reads
    {mapping_summary['UniqueAlignFinalReads'].mean():.0f}\tAverage Final Unique Reads per cell

Mapping Rate
    {mapping_summary['UniqueMappedR1Rate'].mean():.1f}%\tAverage R1 Unique Mapping Rate
    {mapping_summary['UniqueMappedR2Rate'].mean():.1f}%\tAverage R2 Unique Mapping Rate
    {mapping_summary['UniqueClusterMappingRate'].mean():.1f}%\tAverage Cluster Mapping Rate

PCR Duplication
    {mapping_summary['UniqueAlignPCRDuplicationRate'].mean():.1f}%\tAverage PCR Duplication Rate

Contacts
    {mapping_summary['CisContactsRatio'].mean():.2f}\tAverage Cis Contacts Ratio
    {mapping_summary['TransContactsRatio'].mean():.2f}\tAverage Trans Contacts Ratio
    {mapping_summary['MultiContactsRatio'].mean():.2f}\tAverage Multi Contacts Ratio
""")

## mC Fraction

### mCCC

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCCCFrac')

### mCH

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCHFrac')

### mCG

In [ ]:
distplot_and_plate_view(mapping_summary, hue='mCGFrac')

## FASTQ Metric

### TrimmedReadPairs

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TrimmedReadPairs')

## Mapping Rate

### R1 Unique Mapping Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueMappedR1Rate')

### R2 Unique Mapping Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueMappedR2Rate')

### Cluster Mapping Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueClusterMappingRate')

## PCR Duplication Rate

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueAlignPCRDuplicationRate')

## Final Unique Reads

In [ ]:
distplot_and_plate_view(mapping_summary, hue='UniqueAlignFinalReads')

## Chromatin Contacts

### Total Cis Contacts

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TotalCisContacts')

### Total Cis Contacts With Restriction Cut Site

In [ ]:
distplot_and_plate_view(mapping_summary, hue='CisCutContacts')

### Total Trans Contacts

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TotalTransContacts')

### Total Trans Contacts With Restriction Cut Site

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TransCutContacts')

### Cis Contacts Ratio

In [ ]:
distplot_and_plate_view(mapping_summary, hue='CisContactsRatio')

### Trans Contacts Ratio

In [ ]:
distplot_and_plate_view(mapping_summary, hue='TransContactsRatio')

### Multi Contacts Ratio

In [ ]:
distplot_and_plate_view(mapping_summary, hue='MultiContactsRatio')

## Mapping Config

In [ ]:
config